# Lab 2: W&B tracking with a CNN on MNIST

In this notebook, I train a small CNN on the MNIST dataset. I also log metrics, a confusion matrix, sample predictions, checkpoints, and a final model artifact to Weights & Biases. This is different from the original version, which used Fashion-MNIST.

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [2]:
import numpy as np
import wandb

from tensorflow import keras as k
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

In [3]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/vigneshraja/.netrc.
wandb: Currently logged in as: vigneshvrs5 (vigneshvrs5-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
class LogSamplesCallback(k.callbacks.Callback):
    def __init__(self, x, y, labels, max_rows=24):
        super().__init__()
        self.x = x[:max_rows]
        self.y = y[:max_rows]
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x, verbose=0)
        y_true = np.argmax(self.y, axis=1)
        y_pred = np.argmax(preds, axis=1)

        table = wandb.Table(columns=["image", "y_true", "y_pred", "correct", "confidence"])
        for i in range(len(self.x)):
            table.add_data(
                wandb.Image(self.x[i].squeeze()),
                self.labels[y_true[i]],
                self.labels[y_pred[i]],
                bool(y_true[i] == y_pred[i]),
                float(np.max(preds[i])),
            )
        wandb.log({f"samples/epoch_{epoch + 1}": table})


class ConfusionMatrixCallback(k.callbacks.Callback):
    def __init__(self, x_val, y_val, labels):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        cm_plot = wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.labels,
        )
        wandb.log({"validation_confusion_matrix": cm_plot})


class DigitTrainer:
    def __init__(self, project_name="Lab2-mnist-cnn", run_name="mnist_cnn_submission"):
        self.cfg = {
            "epochs": 4,
            "batch_size": 128,
            "learning_rate": 0.001,
            "dropout": 0.25,
            "train_samples": 12000,
            "test_samples": 2500,
            "random_seed": 42,
        }
        k.utils.set_random_seed(self.cfg["random_seed"])
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=self.cfg,
            settings=wandb.Settings(start_method="thread"),
        )
        self.config = wandb.config
        self.labels = [str(i) for i in range(10)]
        self._prepare_data()

    def _prepare_data(self):
        (x_train, y_train), (x_test, y_test) = mnist.load_data()
        x_train = x_train[: self.config.train_samples].astype("float32") / 255.0
        y_train = y_train[: self.config.train_samples]
        x_test = x_test[: self.config.test_samples].astype("float32") / 255.0
        y_test = y_test[: self.config.test_samples]

        self.X_train = x_train[..., None]
        self.X_test = x_test[..., None]
        self.y_train = to_categorical(y_train, num_classes=10)
        self.y_test = to_categorical(y_test, num_classes=10)

    def _build_model(self):
        inputs = k.Input(shape=(28, 28, 1))
        x = k.layers.Conv2D(32, (3, 3), activation="relu")(inputs)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Conv2D(64, (3, 3), activation="relu")(x)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Flatten()(x)
        x = k.layers.Dropout(self.config.dropout)(x)
        x = k.layers.Dense(64, activation="relu")(x)
        outputs = k.layers.Dense(10, activation="softmax")(x)
        model = k.Model(inputs=inputs, outputs=outputs)
        model.compile(
            optimizer=k.optimizers.Adam(learning_rate=self.config.learning_rate),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )
        return model

    def _log_model_artifact(self, model):
        os.makedirs("artifacts", exist_ok=True)
        os.makedirs("checkpoints", exist_ok=True)

        summary_lines = []
        model.summary(print_fn=summary_lines.append)
        with open("artifacts/mnist_model_summary.txt", "w") as file:
            file.write("\n".join(summary_lines))

        model_path = "artifacts/mnist_cnn_model.keras"
        model.save(model_path)

        artifact = wandb.Artifact("mnist_cnn_model", type="model")
        artifact.add_file("artifacts/mnist_model_summary.txt")
        artifact.add_file(model_path)
        self.run.log_artifact(artifact)

    def train(self):
        model = self._build_model()
        callbacks = [
            WandbMetricsLogger(log_freq="epoch"),
            WandbModelCheckpoint("checkpoints/mnist-model-{epoch:02d}.keras", save_weights_only=False),
            LogSamplesCallback(self.X_test, self.y_test, self.labels),
            ConfusionMatrixCallback(self.X_test, self.y_test, self.labels),
            k.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
        ]

        model.fit(
            self.X_train,
            self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=callbacks,
            verbose=1,
        )

        loss, accuracy = model.evaluate(self.X_test, self.y_test, verbose=0)
        wandb.log({"final_loss": loss, "final_accuracy": accuracy})
        self.run.summary["dataset"] = "MNIST"
        self.run.summary["model"] = "Custom CNN"

        self._log_model_artifact(model)
        self.run.finish()


DigitTrainer().train()

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


Epoch 1/4
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.7950 - loss: 0.7224 - val_accuracy: 0.9212 - val_loss: 0.2863
Epoch 2/4
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9473 - loss: 0.1814 - val_accuracy: 0.9516 - val_loss: 0.1615
Epoch 3/4
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9621 - loss: 0.1216 - val_accuracy: 0.9576 - val_loss: 0.1281
Epoch 4/4
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9718 - loss: 0.0897 - val_accuracy: 0.9652 - val_loss: 0.1013


epoch/accuracy,▁▇██
epoch/epoch,▁▃▆█
epoch/learning_rate,▁▁▁▁
epoch/loss,█▂▁▁
epoch/val_accuracy,▁▆▇█
epoch/val_loss,█▃▂▁
final_accuracy,▁
final_loss,▁
dataset,MNIST
epoch/accuracy,0.97183
epoch/epoch,3
